In [1]:
import glob
import os
import pandas as pd
from termcolor import colored, cprint
from datasets import load_dataset

from inspect_ai.log import read_eval_log

/Users/mat/EPFL-courses/RAI/reasoning-blind-spots/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create re-grading dataset

Fetch all samples where grading error occurred, and add them to a JSONL compatible with the re-grading script.

In [2]:
root_dir = "../outputs/"

model_info = pd.read_csv(os.path.join("..", "data", "models_info.csv"))
valids_qids = load_dataset("matsant01/blind-spots-bench", split="test").to_pandas()["QID"].tolist()

In [3]:
def capability_to_qtype(model_type: str):
	if model_type == "VLM":
		return ["text", "multi"]
	elif model_type == "LM" or model_type == "LLM":
		return ["text"]
	elif model_type == "GEN":
		return ["image"]
	else:
		raise ValueError(f"Unknown model type: {model_type}")

In [ ]:

results = []

for model_name, model_type in zip(model_info["model_name"], model_info["model_type"]):
	cleaned_model_name = model_name.split("/")[1] if "/" in model_name else model_name
	qtypes = capability_to_qtype(model_type)

	# Iterate for all folders under outputs/
	for epoch_dir in os.listdir(os.path.join(root_dir)):
		if any(x in epoch_dir for x in [".DS_Store", "validation", "tests"]):
			continue

		for mod in qtypes:
			qtype_subdir = os.path.join(root_dir, epoch_dir, mod)
			if not os.path.exists(qtype_subdir):
				continue

			# Search for the folders matching the model name
			model_dirs = [d for d in os.listdir(qtype_subdir) if d.split("_")[0] == cleaned_model_name]
			if len(model_dirs) == 0:
				cprint(f"No folder found for model {model_name} in {qtype_subdir}", "yellow")
				continue
			elif len(model_dirs) > 1:
				cprint(f"Multiple folders found for model {model_name} in {qtype_subdir}: {model_dirs}", "yellow")
				continue
			model_dir = model_dirs[0]

			# Find the eval log in the model folder
			eval_log_paths = [os.path.join(qtype_subdir, model_dir, f) for f in os.listdir(os.path.join(qtype_subdir, model_dir)) if f.endswith(".eval")]
			if len(eval_log_paths) == 0:
				cprint(f"No eval log found for model {model_name} in {os.path.join(qtype_subdir, model_dir)}", "yellow")
				continue
			elif len(eval_log_paths) > 1:
				cprint(f"Multiple eval logs found for model {model_name} in {os.path.join(qtype_subdir, model_dir)}: {eval_log_paths}", "yellow")
				continue

			# Read the eval log
			eval_log_path = eval_log_paths[0]
			eval_data = read_eval_log(eval_log_path)

			n_epochs = 0
			tmp_results = []
			for sample in eval_data.samples:
				qid = sample.id
				# NOTE: apply here the QID filtering!
				if qid not in valids_qids:
					continue

				scores_list = list(sample.scores.values())
				if len(scores_list) == 0:
					score = 0
					error = True
					continue
				elif len(scores_list) > 1:
					raise ValueError(f"Multiple scores found for sample {qid} in model {model_name}: {scores_list}")
				else:
					score = 1 if scores_list[0] is not None and scores_list[0].value == "C" and sample.error is None else 0
					error = sample.error is not None

					if "possibly exceeded message limit" not in scores_list[0].explanation or score == 1:
						# Only keeping samples to be regraded
						continue


				prompt = sample.input
				solver_answer = sample.output

				epoch = int(sample.epoch)
				n_epochs = max(n_epochs, epoch)

				if "multi" == mod:
					try:
						prompt = sample.input[0].content.text
					except Exception as e:
						prompt = sample.input[0].content[0].text

					image = f"data/images/img_q{int(qid)}"
					# Find extension of the image
					image_path = glob.glob(os.path.join("..", image + ".*"))
					if len(image_path) == 0:
						cprint(f"No image found for sample {qid} in model {model_name} at {image}", "yellow")
						image_path = None
					elif len(image_path) > 1:
						cprint(f"Multiple images found for sample {qid} in model {model_name} at {image}: {image_path}", "yellow")
						image_path = None
					else:
						image_path = os.path.abspath(image_path[0])

				if not error:
					if qtype_subdir.split("/")[-1] in ["text", "multi"]:
						if len(eval_data.stats.model_usage.keys()) >= 1:
							sample.model_usage.pop("google/gemini-3-flash-preview", None)
							usage = list(sample.model_usage.values())
							if len(usage) == 0:
								usage = None
							else: 
								usage = usage[0]
						else:
							usage = sample.role_usage["solver"]
						
						in_tks = usage.input_tokens if usage else 0
						out_tks = usage.output_tokens if usage else 0
						if usage and usage.reasoning_tokens is not None and not "gpt-5" in model_dir.lower():	# NOTE: gpt-5 models count reasoning tokens already as part of output tokens
							out_tks += usage.reasoning_tokens
					else:
						if "gemini" in model_dir.lower():
							in_tks = sample.store["image_generation_metadata"]["usage"]["prompt_token_count"]
							out_tks = sample.store["image_generation_metadata"]["usage"]["total_token_count"]
						elif "gpt" in model_dir.lower():
							in_tks = sample.store["image_generation_metadata"]["usage"]["input_tokens"] if "image_generation_metadata" in sample.store else 0
							out_tks = sample.store["image_generation_metadata"]["usage"]["output_tokens"] if "image_generation_metadata" in sample.store else 0
						else:
							raise ValueError(f"Model at '{model_dir}' not recognized")
				else:
					in_tks = 0
					out_tks = 0



				tmp_results.append({
					"index": qid,
					"prompt": prompt,
					"solution": sample.target,
					"solver_answer": solver_answer.completion,
					"solver_name": model_name,
					"question_type": "text-only" if mod == "text" else "multi-to-text",
					"score": score,
					"error": error,
					"image": image_path if mod == "multi" else None,
					"in_tokens": in_tks,
					"out_tokens": out_tks
				})
			


			results.extend(tmp_results)

Multiple folders found for model deepseek-ai/DeepSeek-V4-Pro in ../outputs/reval/text: ['DeepSeek-V4-Pro_2026-05-02_15-06-38', 'DeepSeek-V4-Pro_2026-05-02_14-43-28']
No folder found for model google/gemma-4-31B-it in ../outputs/reval/text


In [ ]:
len(results)

151

In [19]:
import json

# Save the file re-grade file
with open(os.path.join("..", "data", "re-grade.jsonl"), "w") as f:
	for item in results:
		f.write(json.dumps(item) + "\n")

In the `results_processing.ipynb` notebook, we merge the results from the previous evals and the re-grading ones.

---

### Create sub-type grading dataset

In [17]:
from pathlib import Path

root_dir = "../outputs/first_epoch/image_gen"

model_info = pd.read_csv(os.path.join("..", "data", "models_info.csv"))
dataset_df = load_dataset("matsant01/blind-spots-bench", split="test").to_pandas()

# df[df["multi_subcategory_flag"] == 1]["question_type"].value_counts()
# df.columns
# "multi_subcategory_flag", "Failure mode", "task_type", "sub-task type"


multi_category_ids = dataset_df[dataset_df["multi_subcategory_flag"] == 1]["QID"].tolist()


results = []
for model_dir in os.listdir(os.path.join(root_dir)):
	log_path_pattern = os.path.join(root_dir, model_dir, "*.eval")
	log_path = glob.glob(log_path_pattern)
	if len(log_path) == 0:
		cprint(f"No eval log found for model {model_dir} in {root_dir}", "yellow")
		continue
	elif len(log_path) > 1:
		cprint(f"Multiple eval logs found for model {model_dir} in {root_dir}: {log_path}", "yellow")
		continue

	model_name = model_dir.split("_")[0]

	log_path = log_path[0]
	eval_data = read_eval_log(log_path)


	for sample in eval_data.samples:

		qid = sample.id
		if qid not in multi_category_ids:
			continue

		if isinstance(sample.input, str):
			prompt = sample.input.strip()
		else:
			prompt = sample.input[0].content[0].text

		# Check whether an input image exists:
		in_image_path = "../data/images/img_q" + str(qid)
		in_image_path = glob.glob(in_image_path + ".*")
		if len(in_image_path) == 1:
			in_image_path = os.path.abspath(in_image_path[0])
		else:
			in_image_path = None

		qtype = dataset_df[dataset_df["QID"] == qid]["question_type"].values[0]

		out_image_pattern = os.path.join(root_dir, model_dir, "generated_images", f"generated_{qid}_*.png")
		out_image_path = glob.glob(out_image_pattern)
		if len(out_image_path) == 0:
			cprint(f"No output image found for sample {qid} in model {model_name} at {out_image_path}", "yellow")
			out_image_path = None
		elif len(out_image_path) > 1:
			cprint(f"Multiple output images found for sample {qid} in model {model_name} at {out_image_path}: {out_image_path}", "yellow")
			out_image_path = None
		else:
			out_image_path = out_image_path[0]

		# Make both paths relative to the working directory (upper level of the repo)
		in_image_path = "/".join(in_image_path.split("/")[1:]) if in_image_path else None
		out_image_path = "/".join(out_image_path.split("/")[1:]) if out_image_path else None


		results.append({
			"index": qid,
			"prompt": prompt,
			"solution": sample.target,
			"solver_answer": out_image_path,
			"solver_name": model_name,
			"question_type": qtype,
			"image": in_image_path,
			"failure_mode": dataset_df[dataset_df["QID"] == qid]["failure_mode"].values[0],
			"sub-task_type": dataset_df[dataset_df["QID"] == qid]["sub-task_type"].values[0],
			"task_type": dataset_df[dataset_df["QID"] == qid]["task_type"].values[0],
		})
	



In [21]:
import json
with open(os.path.join("..", "data", "multi_category_grader_inputs.jsonl"), "w") as f:
	for item in results:
		f.write(json.dumps(item) + "\n")